### Setup

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report,
                             f1_score, ConfusionMatrixDisplay)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 100)
RANDOM_STATE = 42

### Reading Data

In [ ]:
df = pd.read_csv("../Data/cumulative.csv")
print("Размер:", df.shape)
df.head()

In [ ]:
df.info(verbose=False)
print("\nClass destribution:")
print(df["koi_disposition"].value_counts(dropna=False))

### Train / Test splitting

In [ ]:
df = df.dropna(subset=["koi_disposition"]).reset_index(drop=True)

X = df.drop(columns=["koi_disposition"])
y = df["koi_disposition"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print("Train:", X_train.shape, " Test:", X_test.shape)
print("Class destribution in Train:\n", y_train.value_counts(normalize=True))

### Visualization

In [ ]:
# Number attributes only
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
print(f"Num of number attributes: {len(num_cols)}")

desc = X_train[num_cols].describe().T
desc["missing_%"] = X_train[num_cols].isna().mean().values * 100
desc.head(15)

In [ ]:
# Plots for some of the key attributes
key_cols = ["koi_period", "koi_prad", "koi_teq", "koi_insol", "koi_model_snr"]
key_cols = [c for c in key_cols if c in num_cols]

fig, axes = plt.subplots(1, len(key_cols), figsize=(20, 4))
for ax, col in zip(axes, key_cols):
    sns.histplot(X_train[col].dropna(), bins=50, ax=ax, kde=True)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# Corr matrix 
corr = X_train[num_cols].corr()
plt.figure(figsize=(14, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Correlation matrix for number-type attributes")
plt.show()

# Top 10 correlated attributes
high_corr = (corr.abs() > 0.5) & (corr.abs() < 1.0)
pairs = [(c1, c2, corr.loc[c1, c2]) for c1 in high_corr.index for c2 in high_corr.columns
         if high_corr.loc[c1, c2] and c1 < c2]
pd.DataFrame(pairs, columns=["col1", "col2", "corr"]).sort_values("corr", key=abs, ascending=False).head(10)